In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

 
DATA_DIR   = Path("./utils")
OUTPUT_DIR = Path("plots")
FMT        = "png"

RESNET_FILES = {
    "none":          "resnet18_lr_results.csv",      
    "flip":          "resnet18_flip_results.csv",
    "blur":          "resnet18_blur_results.csv",
    "jitter":        "resnet18_jitter_results.csv",
    "cutout":        "resnet18_cutout_results.csv",
    "cutmix_alpha_1":"resnet18_cutmix_alpha_1_results.csv",
    "cutmix_alpha_4":"resnet18_cutmix_alpha_4_results.csv",
}

RESNET_BASELINE_FILE = "resnet18_dropout_results.csv"
 

MBV2_FILES = {
    "none":          "experiments_agg_by_config_dp.csv",           
    "flip":          "experiments_agg_by_config_aug.csv",
    "blur":          "experiments_agg_by_config_aug.csv",
    "jitter":        "experiments_agg_by_config_aug.csv",
    "cutout":        "advanced_aug_cutmix_2_agg_by_config.csv",
    "cutmix_alpha_1":"advanced_aug_cutmix_2_agg_by_config.csv",
    "cutmix_alpha_4":"advanced_aug_cutmix_2_agg_by_config.csv",
    "random":        "advanced_random_agg_by_config_random.csv",
}
MBV2_BASELINE_FILE = "experiments_agg_by_config_dp.csv"
 
# Custom CNN files (same schema as ResNet-18 but different column names)
CUSTOM_BASELINE_FILE  = "custom_cnn_phase4_results.csv"   # best regularization
CUSTOM_AUG_FILES = {
    "none":          "custom_cnn_phase5_results.csv",     # aug=none row
    "flip":          "custom_cnn_phase5_results.csv",
    "blur":          "custom_cnn_phase5_results.csv",
    "jitter":        "custom_cnn_phase5_results.csv",
    "cutout":        "custom_cnn_phase7_results.csv",
    "cutmix_alpha_1":"custom_cnn_phase6_results.csv",
    "cutmix_alpha_4":"custom_cnn_phase6_results.csv",
    "random":        "custom_cnn_phase7_results.csv",
}
 
ENSEMBLE_FILE = "ensemble_results.csv"

PALETTE = {
    "Custom CNN":  "#2166ac",
    "ResNet-18":   "#d6604d",
    "MobileNetV2": "#4dac26",
    "Ensemble":    "#8073ac",
}
 
STYLE = {
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": ":",
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
}
 
 
def _best_row(df, col_mean, col_std):
    idx = df[col_mean].idxmax()
    return float(df.loc[idx, col_mean]), float(df.loc[idx, col_std])
 
 
def load_resnet(filename, aug_label=None):
    path = DATA_DIR / filename
    df = pd.read_csv(path)
    return _best_row(df, "test_f1_mean", "test_f1_std")
 
 
def load_mbv2(filename, aug_filter=None):
    path = DATA_DIR / filename
    df = pd.read_csv(path)
    col_mean = "test_f1_macro_eval_mean"
    col_std  = "test_f1_macro_eval_std"
 
    if aug_filter is not None and "aug" in df.columns:
        mask = df["aug"].astype(str).str.lower() == aug_filter.lower()
        df = df[mask]
        if df.empty:
            return None
 
    return _best_row(df, col_mean, col_std)
 
 
def load_custom(filename, aug_filter=None):
    path = DATA_DIR / filename
    df = pd.read_csv(path)
 
    if aug_filter is not None and aug_filter != "none":
        aug_filter_norm = aug_filter.replace("cutmix_alpha_", "cutmix_alpha")
        mask = df["config"].str.contains(aug_filter_norm, case=False, na=False)
        df = df[mask]
        if df.empty:
            return None
 
    if aug_filter == "none":
        mask = df["config"].str.contains("aug=none", case=False, na=False)
        df = df[mask]
        if df.empty:
            return None
 
    return _best_row(df, "test_f1_mean", "test_f1_std")
 
 
def load_ensemble():
    path = DATA_DIR / ENSEMBLE_FILE
    df = pd.read_csv(path)
    return float(df["test_f1_soft"].iloc[0]), float(df["test_f1_hard"].iloc[0])
 

 
AUG_LABELS = [
    "none",
    "flip",
    "blur",
    "jitter",
    "cutout",
    "cutmix_alpha_1",
    "cutmix_alpha_4",
    "random",
]
 
AUG_DISPLAY = {
    "none":           "none (baseline)",
    "flip":           "flip",
    "blur":           "blur",
    "jitter":         "jitter",
    "cutout":         "cutout",
    "cutmix_alpha_1": "cutmix alpha=1",
    "cutmix_alpha_4": "cutmix alpha=4",
    "random":         "random",
}
 
MBV2_AUG_FILTER = {
    "none":           None,             
    "flip":           "flip",
    "blur":           "blur",
    "jitter":         "jitter",
    "cutout":         "cutout",
    "cutmix_alpha_1": "cutmix_alpha_1",
    "cutmix_alpha_4": "cutmix_alpha_4",
    "random":         "random",
}
 
 
def build_aug_table():
    """
    Returns a DataFrame with columns:
      aug, arch, mean, std
    One row per (aug, arch) combination that has data.
    """
    rows = []
 
    for aug in AUG_LABELS:
        if aug in CUSTOM_AUG_FILES:
            result = load_custom(CUSTOM_AUG_FILES[aug], aug_filter=aug)
            if result:
                rows.append({"aug": aug, "arch": "Custom CNN",
                             "mean": result[0], "std": result[1]})
 
        if aug in RESNET_FILES:
            result = load_resnet(RESNET_FILES[aug], aug_label=aug)
            if result:
                rows.append({"aug": aug, "arch": "ResNet-18",
                             "mean": result[0], "std": result[1]})
 
        if aug in MBV2_FILES:
            af = MBV2_AUG_FILTER.get(aug)
            result = load_mbv2(MBV2_FILES[aug], aug_filter=af)
            if result:
                rows.append({"aug": aug, "arch": "MobileNetV2",
                             "mean": result[0], "std": result[1]})
 
    return pd.DataFrame(rows)
 
 
def build_summary():
    aug_df = build_aug_table()
 
    summary = {}
    for arch in ["Custom CNN", "ResNet-18", "MobileNetV2"]:
        sub = aug_df[aug_df["arch"] == arch]
        baseline_row = sub[sub["aug"] == "none"]
        baseline = (float(baseline_row["mean"].iloc[0]),
                    float(baseline_row["std"].iloc[0])) if not baseline_row.empty else None
        best_idx = sub["mean"].idxmax()
        best = (float(sub.loc[best_idx, "mean"]), float(sub.loc[best_idx, "std"]))
        summary[arch] = {"baseline": baseline, "best": best}
 
    return summary
 
 
def save(fig, name):
    OUTPUT_DIR.mkdir(exist_ok=True)
    path = OUTPUT_DIR / f"{name}.{FMT}"
    fig.savefig(path)
    print(f"  v {path}")
    plt.close(fig)
 
 
def plot_arch_summary():
    archs   = ["Custom CNN", "ResNet-18", "MobileNetV2"]
    summary = build_summary()
    soft_f1, hard_f1 = load_ensemble()
 
    base_vals = [summary[a]["baseline"][0] for a in archs]
    base_stds = [summary[a]["baseline"][1] for a in archs]
    best_vals = [summary[a]["best"][0]     for a in archs]
    best_stds = [summary[a]["best"][1]     for a in archs]
 
    x     = np.arange(len(archs))
    width = 0.32
 
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(10, 5.5))
 
        bars_base = ax.bar(x - width/2, base_vals, width, yerr=base_stds,
                           capsize=5,
                           color=[PALETTE[a] for a in archs], alpha=0.38,
                           error_kw=dict(lw=1.3, capthick=1.3),
                           label="No augmentation (best regularization)")
 
        bars_best = ax.bar(x + width/2, best_vals, width, yerr=best_stds,
                           capsize=5,
                           color=[PALETTE[a] for a in archs], alpha=0.90,
                           error_kw=dict(lw=1.3, capthick=1.3),
                           label="Best augmentation")
 
        for bar, v, s in zip(list(bars_base) + list(bars_best),
                              base_vals + best_vals,
                              base_stds + best_stds):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + s + 0.018,
                    f"{v:.4f}", ha="center", fontsize=9)
 
        for i, a in enumerate(archs):
            gain = summary[a]["best"][0] - summary[a]["baseline"][0]
            ax.annotate(f"+{gain:.4f}",
                        xy=(x[i] + width/2,
                            summary[a]["best"][0] + summary[a]["best"][1] + 0.052),
                        ha="center", fontsize=9,
                        color=PALETTE[a], fontweight="bold")
 
        ax.axhline(soft_f1, color=PALETTE["Ensemble"], lw=1.8, ls="--",
                   label=f"Ensemble soft voting ({soft_f1:.4f})")
        ax.axhline(hard_f1, color=PALETTE["Ensemble"], lw=1.8, ls=":",
                   label=f"Ensemble hard voting ({hard_f1:.4f})")
 
        ax.set_xticks(x)
        ax.set_xticklabels(archs, fontsize=12)
        ax.set_ylabel("Test Macro F1")
        ax.set_title("Architecture comparison — Test Macro F1 (mean ± std, n=5 seeds)")
        ax.set_ylim(0, max(best_vals) + 0.10)
 
        legend_handles = []
        for a in archs:
            legend_handles.append(
                mpatches.Patch(facecolor=PALETTE[a], alpha=0.38, edgecolor=PALETTE[a],
                               label=f"{a} — no augmentation"))
            legend_handles.append(
                mpatches.Patch(facecolor=PALETTE[a], alpha=0.90, edgecolor=PALETTE[a],
                               label=f"{a} — best augmentation"))
        legend_handles += [
            plt.Line2D([0],[0], color=PALETTE["Ensemble"], lw=1.8, ls="--",
                       label="Ensemble (soft)"),
            plt.Line2D([0],[0], color=PALETTE["Ensemble"], lw=1.8, ls=":",
                       label="Ensemble (hard)"),
        ]
        ax.legend(handles=legend_handles, fontsize=9, loc="upper left",
                  bbox_to_anchor=(1.01, 1), borderaxespad=0, ncol=1)
 
        fig.tight_layout()
        save(fig, "07_architecture_summary")
 

 
def plot_aug_comparison():
    aug_df  = build_aug_table()
    archs   = ["Custom CNN", "ResNet-18", "MobileNetV2"]
    x_pos   = {a: i for i, a in enumerate(archs)}
 
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 6))
 
        for row_i, aug in enumerate(AUG_LABELS):
            sub = aug_df[aug_df["aug"] == aug]
            for _, row in sub.iterrows():
                arch  = row["arch"]
                x     = x_pos[arch]
                color = PALETTE[arch]
                ax.scatter(x, row_i, s=180, color=color, zorder=3,
                           alpha=0.85, edgecolors="white", linewidths=0.8)
                ax.text(x + 0.13, row_i, f"{row['mean']:.4f}",
                        va="center", fontsize=9, color=color)

        baseline_row = AUG_LABELS.index("none")
        ax.axhspan(baseline_row - 0.45, baseline_row + 0.45,
                   color="gray", alpha=0.08, zorder=0)
 
        ax.set_yticks(range(len(AUG_LABELS)))
        ax.set_yticklabels([AUG_DISPLAY[a] for a in AUG_LABELS], fontsize=11)
        ax.set_xticks(range(len(archs)))
        ax.set_xticklabels(archs, fontsize=11)
        ax.set_xlim(-0.5, len(archs) - 0.15)
        ax.invert_yaxis()
        ax.set_title("Test Macro F1 by augmentation strategy and architecture\n"
                     "(shaded row = no augmentation baseline)")
        ax.grid(axis="x", alpha=0.0)
        ax.grid(axis="y", alpha=0.25, linestyle=":")
 
        legend_handles = [mpatches.Patch(color=PALETTE[a], label=a) for a in archs]
        ax.legend(handles=legend_handles, loc="lower right", fontsize=10)
 
        fig.tight_layout()
        save(fig, "08_augmentation_comparison")
 

In [12]:
print(f"Saving plots to: {OUTPUT_DIR.resolve()}\n")
print("7/8  Architecture summary...")
plot_arch_summary()
print("8/8  Augmentation comparison...")
plot_aug_comparison()
print("\nDone!")

Saving plots to: C:\Semestr_8\DeepLearning\repo\DeepLearningProject1\plots

7/8  Architecture summary...
  v plots\07_architecture_summary.png
8/8  Augmentation comparison...
  v plots\08_augmentation_comparison.png

Done!
